In [1]:
from pathlib import Path
import sys

# ---- Project root setup ----

PROJECT_ROOT = Path.cwd().parents[0]
sys.path.insert(0, str(PROJECT_ROOT))

# ---- Set plan filename ----

from spc_agent.agent.agent_runner import ask_agent

In [2]:
# known good run - to set up replot tests.

result = ask_agent(
    prompt="CPR11 needed maintenance last week due to motor temperature and again due to vibration. How is the tool doing now?",
    project_root=PROJECT_ROOT,
    planner_backend="llm",
    show_json=True,
)

print(result.run_summary_path)


/Users/mcmoore/anaconda_projects/GitHub/agentic-predictive-maintenance/runs/2026-03-11T12-37-29/run_summary.md


In [3]:
# Replot: latest - will fail. matching job_id not found.
from runner.replot_run import replot_from_plan

plan_filename = "test_replot_latest.json"
replot_plan = json.loads((PROJECT_ROOT / "planner" / plan_filename).read_text())

replot_dir_output = replot_from_plan(replot_plan, PROJECT_ROOT)
print(replot_dir_output)


❌ Replot aborted: Replot failed: job_id 'arm_vibration_7d' was not found in the referenced run.
Available job_ids: ['cpr11_motor_temp_status', 'cpr11_vibration_status']
Resolved run directory: /Users/mcmoore/anaconda_projects/GitHub/agentic-predictive-maintenance/runs/2026-03-11T12-37-29
[]


In [4]:
# Replot: latest + job_id - will run
from runner.replot_run import replot_from_plan

plan_filename = "test_replot_latest_job_id.json"
replot_plan = json.loads((PROJECT_ROOT / "planner" / plan_filename).read_text())

replot_dir_output = replot_from_plan(replot_plan, PROJECT_ROOT)
print(replot_dir_output)


[PosixPath('/Users/mcmoore/anaconda_projects/GitHub/agentic-predictive-maintenance/runs/2026-03-11T12-34-29/arm_vibration_7d/replots/2026-03-11T12-37-29')]


In [5]:
# known good run - ARM.
result = ask_agent(
    prompt="Plot 7 days of vibration data for ARM tools.",
    project_root=PROJECT_ROOT,
    planner_backend="llm",
)

print(result.run_summary_path)
print(result.verification_summary)
print('---------------')
print('Planner Output')
print('---------------')
print(result.planner_raw_output)

/Users/mcmoore/anaconda_projects/GitHub/agentic-predictive-maintenance/runs/2026-03-11T12-37-32/run_summary.md
✅ All artifacts verified (base + replots).
---------------
Planner Output
---------------
{
"runs": [
{
"run_id": "arm_vibration_7d",
"request_text": "Plot 7 days of vibration data for ARM tools.",
"jobs": [
{
"job_id": "arm_vibration_7d",
"sql_template": "fleet_sensor_history",
"preprocess": "ewma_spc",
"filters": {
"entity_group": "ARM",
"entity": null,
"sensor": "vibration_rms",
"start_ts": "2024-01-08T00:00:00",
"end_ts": "2024-01-15T00:00:00"
},
"outputs": {
"plots": [
{
"plot": "fleet_time_trend",
"plot_name": "arm_vibration_7d.png"
}
]
}
}
]
}
]
}


In [6]:
# Replot: latest - will run.
from runner.replot_run import replot_from_plan

plan_filename = "test_replot_latest.json"
replot_plan = json.loads((PROJECT_ROOT / "planner" / plan_filename).read_text())

replot_dir_output = replot_from_plan(replot_plan, PROJECT_ROOT)
print(replot_dir_output)


[PosixPath('/Users/mcmoore/anaconda_projects/GitHub/agentic-predictive-maintenance/runs/2026-03-11T12-37-32/arm_vibration_7d/replots/2026-03-11T12-37-32')]


In [7]:
# Replot: non-existent. will fail
from runner.replot_run import replot_from_plan

plan_filename = "test_replot_nonexist_job_id.json"
replot_plan = json.loads((PROJECT_ROOT / "planner" / plan_filename).read_text())

replot_dir_output = replot_from_plan(replot_plan, PROJECT_ROOT)
print(replot_dir_output)


❌ Replot aborted: No prior run found containing job_id='does_not_exist'.
[]


## Results

- Replot: latest - will fail. matching job_id not found. ✅
- Replot: latest + job_id - will run ✅
- Replot: latest - will run. ✅
- Replot: non-existent. will fail ✅
